# Train Model B — GPA Regressor

The original Model B (`modelB.pkl`) used `GradientBoostingRegressor` from sklearn — **not supported by m2cgen 0.10** for JavaScript transpilation.

This notebook retrains Model B using `XGBRegressor` (supported by m2cgen) on the same dataset and split, so performance can be compared fairly.

Output: `models/modelB.pkl` (~565 KB)

## 1. Imports & Config

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import joblib
import warnings
from scipy import stats

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Adjust dataset path if needed
DATA_PATH = os.path.join(REPO_ROOT, "..", "Datasets", "student_lifestyle_dataset.csv")
OUT_PATH  = os.path.join(REPO_ROOT, "models", "modelB.pkl")

# Column order MUST match src.config.MODEL_B_BASE_FEATURES
FEATURE_ORDER = [
    "study_hours", "eca_hours", "sleep_hours",
    "social_hours", "physical_hours", "stress_level",
]

## 2. Load & Preprocess Data

In [2]:
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=["Student_ID"])
df = df.rename(columns={
    "Study_Hours_Per_Day":              "study_hours",
    "Extracurricular_Hours_Per_Day":    "eca_hours",
    "Sleep_Hours_Per_Day":              "sleep_hours",
    "Social_Hours_Per_Day":             "social_hours",
    "Physical_Activity_Hours_Per_Day":  "physical_hours",
    "Stress_Level":                     "stress_level",
    "GPA":                              "gpa",
})
df["stress_level"] = df["stress_level"].map({"Low": 0, "Moderate": 1, "High": 2})

# Z-score cleaning — same as original modelB
df = df[(np.abs(stats.zscore(df.select_dtypes("number"))) < 3).all(axis=1)]

print(f"Dataset after cleaning: {len(df):,} rows")
df.head()

Dataset after cleaning: 1,996 rows


,study_hours,eca_hours,sleep_hours,social_hours,physical_hours,gpa,stress_level
0,6.9,3.8,8.7,2.8,1.8,2.99,1
1,5.3,3.5,8.0,4.2,3.0,2.75,0
2,5.1,3.9,9.2,1.2,4.6,2.67,0
3,6.5,2.1,7.2,1.7,6.5,2.88,1
4,8.1,0.6,6.5,2.2,6.6,3.51,2


## 3. Train / Test Split

In [3]:
X = df[FEATURE_ORDER]
y = df["gpa"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

Train: 1,397 | Test: 599


## 4. Training

`n_estimators` is capped at ≤ 600 to prevent m2cgen from hitting Python's recursion limit when building the AST during transpilation.

In [4]:
model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=3,
    learning_rate=0.035,
    subsample=0.9,
    colsample_bytree=0.8,
    reg_lambda=1.5,
    min_child_weight=3,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)
model.fit(X_train, y_train)
pred = model.predict(X_test)

## 5. Evaluation

In [5]:
print("=== MODEL B (XGBRegressor) — TEST METRICS ===")
print(f"R2   : {r2_score(y_test, pred):.4f}")
print(f"MAE  : {mean_absolute_error(y_test, pred):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, pred)):.4f}")

=== MODEL B (XGBRegressor) — TEST METRICS ===
R2   : 0.5075
MAE  : 0.1662
RMSE : 0.2085


## 5b. Cross-Validation

5-fold KFold R² — confirms the compact regressor generalizes across splits.

In [6]:
# 5-fold cross-validation (R2)
from sklearn.model_selection import KFold, cross_val_score

def _build_model():
    return xgb.XGBRegressor(
        n_estimators=500, max_depth=3, learning_rate=0.035, subsample=0.9,
        colsample_bytree=0.8, reg_lambda=1.5, min_child_weight=3,
        random_state=42, n_jobs=-1, verbosity=0,
    )

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(_build_model(), X, y, cv=kf, scoring="r2", n_jobs=-1)
print("=== COMPACT MODEL B — CROSS-VALIDATED (5-fold) ===")
print(f"CV R2 : {scores.mean():.4f} +/- {scores.std():.4f}")


=== COMPACT MODEL B — CROSS-VALIDATED (5-fold) ===
CV R2 : 0.5097 +/- 0.0331


## 6. Save Model

In [7]:
joblib.dump(model, OUT_PATH)
size_kb = os.path.getsize(OUT_PATH) / 1024
print(f"Saved {OUT_PATH} ({size_kb:.1f} KB)")
print(f"Feature order: {FEATURE_ORDER}")

Saved E:\BINUS\Semester 4\Machine Learning\AOL\Academic-Shield-AI-Based-Burnout-Meter-and-Student-Performance-Predictor\models\modelB.pkl (562.3 KB)
Feature order: ['study_hours', 'eca_hours', 'sleep_hours', 'social_hours', 'physical_hours', 'stress_level']
